In [1]:
import os
import sys
import pandas as pd
import joblib
import duckdb
from datetime import timedelta

# Layihə modullarını tanımaq üçün src qovluğunu yola əlavə edirik
sys.path.append(os.path.join(os.getcwd(), 'src'))

import database
import pipeline
from config import CITIES

def generate_city_forecasts():
    # 1. Pipeline-ı işə salırıq (Ən son datanı çəkir və feature-ları hazırlayır)
    print("🚀 Pipeline işə düşür (Historical + Forecast data çəkilir)...")
    pipeline.run_pipeline(mode="incremental")
    
    # 2. Database bağlantısını qururuq
    conn = database.get_connection()
    
    # 3. Modellərin yerləşdiyi qovluq
    model_dir = "ML"
    
    # Şəhərlər üzrə modellərin adlarını təyin edirik
    city_models = {
        "Baku": "BAKU_0_7_xgb.pkl",
        "Lenkeran": "LENKERAN_0_7_xgb.pkl",
        "Quba": "QUBA_0_7_xgb.pkl",
        "Saatli": "SAATLI_0_7_xgb.pkl",
        "Zerdab": "ZERDAB_0_7_xgb.pkl"
    }

    all_forecast_results = []

    for city in CITIES:
        city_name = city['name']
        model_path = os.path.join(model_dir, city_models.get(city_name, ""))
        
        if not os.path.exists(model_path):
            print(f"⚠️ Model tapılmadı: {city_name} üçün {model_path} yoxdur.")
            continue

        print(f"🔮 {city_name} üçün proqnoz hazırlanır...")
        
        # Modeli yükləyirik
        model = joblib.load(model_path)
        
        # Modellərin öyrəndiyi feature ardıcıllığını alırıq
        model_features = model.get_booster().feature_names

        # Proqnoz (Forecast) datalarını götürürük
        df_features = conn.execute(f"""
            SELECT * FROM analytics.weather_features 
            WHERE city = '{city_name}' AND data_type = 'forecast'
            ORDER BY date
        """).df()

        # Sonuncu məlum olan rütubəti (Historical datadan) alırıq ki, ilk proqnoza başlayaq
        last_real_moisture = conn.execute(f"""
            SELECT soil_moisture_0_to_7cm_mean FROM analytics.weather_features 
            WHERE city = '{city_name}' AND data_type = 'historical'
            ORDER BY date DESC LIMIT 1
        """).fetchone()[0]

        current_prev_moisture = last_real_moisture
        city_forecasts = []

        # 14 günlük dövr (Recursive Prediction)
        for i, row in df_features.iterrows():
            # Cari günün sətiri
            input_row = row.to_frame().get(i).to_dict()
            
            # prev_soil_moisture-ı yeniləyirik (bir əvvəlki günün proqnozu və ya son real data)
            input_row['prev_soil_moisture'] = current_prev_moisture
            
            # Modellə eyni formatda DataFrame yaradırıq
            X_input = pd.DataFrame([input_row])[model_features]
            
            # Proqnoz veririk
            prediction = model.predict(X_input)[0]
            
            # Növbəti addım üçün proqnozu 'prev' kimi yadda saxlayırıq
            current_prev_moisture = prediction
            
            city_forecasts.append({
                "City": city_name,
                "Date": row['date'],
                "Predicted_Soil_Moisture": round(float(prediction), 4)
            })

        all_forecast_results.extend(city_forecasts)

    # 4. Nəticələri göstəririk
    final_df = pd.DataFrame(all_forecast_results)
    print("\n✅ 14 Günlük Torpaq Rütubəti Proqnozu:")
    print(final_df.pivot(index='Date', columns='City', values='Predicted_Soil_Moisture'))
    
    # Nəticəni CSV kimi saxlamaq istəsəniz:
    # final_df.to_csv("forecast_results.csv", index=False)
    
    conn.close()

if __name__ == "__main__":
    generate_city_forecasts()

ModuleNotFoundError: No module named 'database'

In [3]:
%run ../src/run_forecast.py

--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\ASUS\anaconda3\envs\ironhack\Lib\logging\__init__.py", line 1154, in emit
    stream.write(msg + self.terminator)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\ASUS\anaconda3\envs\ironhack\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\U0001f680' in position 33: character maps to <undefined>
Call stack:
  File "C:\Users\ASUS\anaconda3\envs\ironhack\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\ASUS\anaconda3\envs\ironhack\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\ASUS\anaconda3\envs\ironhack\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\ASUS\a

🚀 Pipeline işə düşür...
🔄 Cleaning data from raw.weather_daily ...
📥 Raw data loaded: 120 rows, 15 columns

📅 Date continuity summary:
       city start_date   end_date  expected_days  actual_days  missing_days  \
0      Baki 2026-04-18 2026-05-11             24           24             0   
1  Lenkeran 2026-04-18 2026-05-11             24           24             0   
2      Quba 2026-04-18 2026-05-11             24           24             0   
3    Saatli 2026-04-18 2026-05-11             24           24             0   
4    Zerdab 2026-04-18 2026-05-11             24           24             0   

  missing_list  
0           []  
1           []  
2           []  
3           []  
4           []  

✅ Cleaned data written to staging.weather_daily
📊 Sample of cleaned data:
city       date  temperature_2m_mean  et0_fao_evapotranspiration_sum  sunshine_duration  shortwave_radiation_sum  relative_humidity_2m_mean  surface_pressure_mean  precipitation_sum  precipitation_hours  wind_spee